# 02. PE label and prior audit

Audit the released PE labels and exact PE-prior information needed for population reweighting. Start with `GW230627_015337`; do not choose labels using ESS or the eventual population score.


In [26]:
from pathlib import Path
import json
import os

import h5py
import numpy as np
import pandas as pd
import bilby
from bilby.gw.cosmology import get_cosmology
from astropy.constants import c

EVENT_ID = "GW230627_015337"
inventory_dir = Path(os.environ["GWTC_RESULTS_ROOT"]) / "pilot" / "inventory"
index = json.loads((inventory_dir / "pe_release_index.json").read_text())
assert index["ready_for_label_audit"]


## 1. Released candidate labels

Enumerate the available results before applying any label-selection rule.


In [2]:
candidate_rows = []
for release, payload in index["releases"].items():
    if not payload.get("present", False):
        continue
    for event_id, event in payload["events"].items():
        for row in event["rows"]:
            candidate_rows.append({"release": release, "event_id": event_id, **row})

candidates = pd.DataFrame(candidate_rows)
event_candidates = candidates[candidates["event_id"] == EVENT_ID].copy()

event_candidates[[
    "release",
    "event_id",
    "result_samples_key",
    "mass_1_source_median",
    "mass_2_source_median",
    "redshift_median",
]]


,release,event_id,result_samples_key,mass_1_source_median,mass_2_source_median,redshift_median
0,o4a,GW230627_015337,C00:IMRPhenomXPHM-SpinTaylor,8.483478,5.714881,0.066302
1,o4a,GW230627_015337,C00:Mixed,8.449413,5.742252,0.065766
2,o4a,GW230627_015337,C00:SEOBNRv5PHM,8.422318,5.762235,0.065132


## 2. Deep PESummary audit

Confirm sample counts and availability of the coordinates needed for PE-prior reconstruction.


In [3]:
deep_path = inventory_dir / f"pe_{EVENT_ID}_deep.json"
deep_report = json.loads(deep_path.read_text())

required = {
    "mass_1_source", "mass_2_source", "mass_ratio", "redshift",
    "luminosity_distance", "mass_1", "mass_2",
    "a_1", "a_2", "tilt_1", "tilt_2", "chi_eff", "chi_p",
}

rows = []
for analysis in deep_report["analyses"]:
    params = set(analysis["parameters"])
    rows.append({
        "label": analysis["label"],
        "sample_count": analysis["sample_count"],
        "parameter_count": len(params),
        "all_required_coordinates": required <= params,
    })

pd.DataFrame(rows)


,label,sample_count,parameter_count,all_required_coordinates
0,C00:IMRPhenomXPHM-SpinTaylor,16191,149,True
1,C00:Mixed,32382,149,True
2,C00:SEOBNRv5PHM,16577,149,True


## 3. Analysis metadata and mixed-posterior composition

Inspect only the metadata needed to understand the candidate labels.


In [4]:
shallow_report = json.loads(
    (inventory_dir / f"pe_{EVENT_ID}_shallow.json").read_text()
)
pe_path = Path(shallow_report["path"])
labels = [
    "C00:IMRPhenomXPHM-SpinTaylor",
    "C00:Mixed",
    "C00:SEOBNRv5PHM",
]

metadata_paths = {
    "approximant": "approximant",
    "preferred": "meta_data/other/preferred",
    "pe_algorithm": "meta_data/sampler/pe_algorithm",
    "sampler": "config_file/config/sampler",
    "cosmology": "config_file/config/cosmology",
    "default_prior": "config_file/config/default-prior",
}

def decode_value(value):
    arr = np.asarray(value).reshape(-1)
    if len(arr) != 1:
        return str(value)
    value = arr[0]
    return value.decode() if isinstance(value, (bytes, np.bytes_)) else value

metadata_rows = []
with h5py.File(pe_path, "r") as f:
    for label in labels:
        row = {"label": label}
        for name, subpath in metadata_paths.items():
            path = f"{label}/{subpath}"
            row[name] = decode_value(f[path][()]) if path in f else None
        metadata_rows.append(row)

    mixed_components = decode_value(
        f["C00:Mixed/meta_data/meta_data/combined"][()]
    )

print("C00:Mixed combines:", mixed_components)
pd.DataFrame(metadata_rows)


C00:Mixed combines: C00:IMRPhenomXPHM-SpinTaylor, C00:SEOBNRv5PHM


,label,approximant,preferred,pe_algorithm,sampler,cosmology,default_prior
0,C00:IMRPhenomXPHM-SpinTaylor,IMRPhenomXPHM,False,bilby,dynesty,Planck15_lal,BBHPriorDict
1,C00:Mixed,None,None,None,None,None,None
2,C00:SEOBNRv5PHM,SEOBNRv5PHM,False,bilby,dynesty,Planck15_lal,BBHPriorDict


## 4. PE-prior consistency across component analyses

`C00:Mixed` combines the two waveform-specific analyses. Verify that their stored analytic prior definitions are identical before treating the mixture as sharing one PE prior.


In [5]:
with h5py.File(pe_path, "r") as f:
    label1 = "C00:IMRPhenomXPHM-SpinTaylor"
    label2 = "C00:SEOBNRv5PHM"
    p1 = f[f"{label1}/priors/analytic"]
    p2 = f[f"{label2}/priors/analytic"]

    names1, names2 = set(p1.keys()), set(p2.keys())
    common = sorted(names1 & names2)
    differences = [
        name for name in common
        if not np.array_equal(p1[name][()], p2[name][()])
    ]

prior_check = {
    "only_in_imrphenom": sorted(names1 - names2),
    "only_in_seobnr": sorted(names2 - names1),
    "common_prior_entries": len(common),
    "different_prior_entries": len(differences),
}
prior_check

{'only_in_imrphenom': [],
 'only_in_seobnr': [],
 'common_prior_entries': 77,
 'different_prior_entries': 0}

## 5. Current decision record — `GW230627_015337`

**Candidate labels**
- `C00:IMRPhenomXPHM-SpinTaylor`
- `C00:Mixed`
- `C00:SEOBNRv5PHM`

**File-level findings**
- All three labels contain the required population coordinates.
- `C00:Mixed` combines the IMRPhenomXPHM-SpinTaylor and SEOBNRv5PHM analyses.
- The two component analyses use Bilby/Dynesty, `Planck15_lal`, and `BBHPriorDict`.
- Their stored analytic prior definitions match exactly: 77 common entries, 0 differences.

**Frozen selection principle**
Use the official release-designated default PE result; do not select a label using sample count, ESS, or the eventual population score.

**Selected label for this event**
`C00:Mixed`, subject to retaining the official release documentation as provenance for the default-PE designation.


## 6. Next task: exact PE-prior reconstruction

Reconstruct `log_pe_prior` in the population coordinates required by the released GWTC-3 models. Document the sampled coordinates, support, normalization, cosmology, transformations, and Jacobians. Validate against the stored analytic prior definitions and prior samples before writing the `EventPosterior` contract.


In [13]:
with h5py.File(pe_path, "r") as f:
    samples = f[f"{label}/posterior_samples"][()]

m1_det = samples["mass_1"]
m2_det = samples["mass_2"]

m1_src = samples["mass_1_source"]
m2_src = samples["mass_2_source"]

z = samples["redshift"]
chirp_stored = samples["chirp_mass"]

In [14]:
def chirp_mass(m1, m2):
    return (m1 * m2)**(3/5) / (m1 + m2)**(1/5)

chirp_from_detector = chirp_mass(m1_det, m2_det)
chirp_from_source = chirp_mass(m1_src, m2_src)

print(
    "stored chirp_mass matches detector-frame:",
    np.allclose(chirp_stored, chirp_from_detector)
)

print(
    "stored chirp_mass matches source-frame:",
    np.allclose(chirp_stored, chirp_from_source)
)

print(
    "stored chirp_mass = (1+z)*source chirp mass:",
    np.allclose(
        chirp_stored,
        (1 + z) * chirp_from_source
    )
)

stored chirp_mass matches detector-frame: True
stored chirp_mass matches source-frame: False
stored chirp_mass = (1+z)*source chirp mass: True


In [12]:
print(
    "max |stored - detector|:",
    np.max(np.abs(chirp_stored - chirp_from_detector))
)

print(
    "max |stored - source|:",
    np.max(np.abs(chirp_stored - chirp_from_source))
)

print(
    "max |stored - (1+z)*source|:",
    np.max(
        np.abs(
            chirp_stored - (1 + z) * chirp_from_source
        )
    )
)

max |stored - detector|: 3.552713678800501e-15
max |stored - source|: 0.5310904092986437
max |stored - (1+z)*source|: 4.440892098500626e-15


In [18]:
print(
    pd.DataFrame(metadata_rows)[
        ["label", "cosmology"]
    ]
)

                          label     cosmology
0  C00:IMRPhenomXPHM-SpinTaylor  Planck15_lal
1                     C00:Mixed          None
2               C00:SEOBNRv5PHM  Planck15_lal


In [19]:
def ddL_dz(z, cosmology, dz=1e-5):
    z = np.asarray(z, dtype=float)

    dL_plus = cosmology.luminosity_distance(z + dz).value
    dL_minus = cosmology.luminosity_distance(z - dz).value

    return (dL_plus - dL_minus) / (2 * dz)

In [24]:
z_test = 0.05

print(ddL_dz(z_test, cosmology))

4746.078590908098


In [23]:
from bilby.gw.cosmology import get_cosmology

cosmology = get_cosmology("Planck15_lal")

print(cosmology)

FlatLambdaCDM(name="Planck15_LAL", H0=67.9 km / (Mpc s), Om0=0.3065, Tcmb0=0.0 K, Neff=3.04, m_nu=None, Ob0=0.0)


In [25]:
print("name:", cosmology.name)
print("H0:", cosmology.H0)
print("Om0:", cosmology.Om0)
print("luminosity distance at z=0.05:",
      cosmology.luminosity_distance(0.05))

name: Planck15_LAL
H0: 67.9 km / (Mpc s)
Om0: 0.3065
luminosity distance at z=0.05: 229.10821725669672 Mpc


In [27]:

def ddL_dz_analytic(z, cosmology):
    Dc = cosmology.comoving_distance(z)

    return (
        Dc
        + (1 + z) * c / cosmology.H(z)
    ).to("Mpc")

In [28]:
z_test = 0.05

numeric = ddL_dz(z_test, cosmology)
analytic = ddL_dz_analytic(z_test, cosmology).value

print("numeric :", numeric)
print("analytic:", analytic)
print("difference:", numeric - analytic)
print("relative difference:", abs(numeric - analytic) / analytic)

numeric : 4746.078590908098
analytic: 4746.078591186741
difference: -2.786428012768738e-07
relative difference: 5.87101110787127e-11


In [29]:
def pe_coordinate_jacobian(m1_source, m2_source, z, cosmology):
    q = m2_source / m1_source

    mass_factor = (
        (1 + z)
        * q**(3/5)
        / (m1_source * (1 + q)**(1/5))
    )

    distance_factor = ddL_dz_analytic(
        z,
        cosmology
    ).value

    return mass_factor * distance_factor

In [30]:
J = pe_coordinate_jacobian(
    m1_src,
    m2_src,
    z,
    cosmology
)

print("Number of samples:", len(J))
print("All finite:", np.all(np.isfinite(J)))
print("All positive:", np.all(J > 0))

print("min J:", np.min(J))
print("median J:", np.median(J))
print("max J:", np.max(J))

print("\nFirst five:")
print(J[:5])

Number of samples: 32382
All finite: True
All positive: True
min J: 155.12293060701046
median J: 432.3667333147721
max J: 685.8620171540547

First five:
[298.83095107 478.32712027 380.81893363 536.83630796 558.62873789]


In [31]:
J_alt = (
    chirp_stored
    / m1_src**2
    * ddL_dz_analytic(z, cosmology).value
)

print(
    "Two Jacobian expressions agree:",
    np.allclose(J, J_alt)
)

print(
    "max absolute difference:",
    np.max(np.abs(J - J_alt))
)

print(
    "max relative difference:",
    np.max(np.abs(J - J_alt) / J)
)

Two Jacobian expressions agree: True
max absolute difference: 4.547473508864641e-13
max relative difference: 7.070645219995715e-16


In [33]:
from bilby.gw.prior import (
    UniformInComponentsChirpMass,
    UniformInComponentsMassRatio,
    UniformSourceFrame,
)

import inspect

print(inspect.getsource(UniformInComponentsChirpMass))
print(inspect.getsource(UniformInComponentsMassRatio))
print(inspect.getsource(UniformSourceFrame))

class UniformInComponentsChirpMass(PowerLaw):
    r"""
    Prior distribution for chirp mass which is uniform in component masses.

    This is useful when chirp mass and mass ratio are sampled while the
    prior is uniform in component masses.

    .. math::

        p({\cal M}) \propto {\cal M}

    Notes
    -----
    This prior is intended to be used in conjunction with the corresponding
    :code:`bilby.gw.prior.UniformInComponentsMassRatio`.
    """

    def __init__(self, minimum, maximum, name='chirp_mass',
                 latex_label=r'$\mathcal{M}$', unit=None, boundary=None):
        """
        Parameters
        minimum : float
            The minimum of chirp mass
        maximum : float
            The maximum of chirp mass
        name: see superclass
        latex_label: see superclass
        unit: see superclass
        boundary: see superclass
        """
        super(UniformInComponentsChirpMass, self).__init__(
            alpha=1., minimum=minimum, maximum=max

In [34]:
from bilby.gw.prior import (
    UniformInComponentsChirpMass,
    UniformInComponentsMassRatio,
    UniformSourceFrame,
)

chirp_prior = UniformInComponentsChirpMass(
    minimum=6.38537095164235,
    maximum=6.4428834107576165,
    name="chirp_mass",
)

q_prior = UniformInComponentsMassRatio(
    minimum=0.05,
    maximum=1.0,
    name="mass_ratio",
)

distance_prior = UniformSourceFrame(
    minimum=50.0,
    maximum=471.599,
    cosmology=cosmology,
    name="luminosity_distance",
)

In [35]:
q = m2_det / m1_det

with h5py.File(pe_path, "r") as f:
    samples = f["C00:Mixed/posterior_samples"][()]
    
dL = samples["luminosity_distance"]

In [36]:
p_chirp = chirp_prior.prob(chirp_stored)
p_q = q_prior.prob(q)
p_dL = distance_prior.prob(dL)

print("chirp finite:", np.all(np.isfinite(p_chirp)))
print("q finite:", np.all(np.isfinite(p_q)))
print("distance finite:", np.all(np.isfinite(p_dL)))

print("chirp positive:", np.all(p_chirp > 0))
print("q positive:", np.all(p_q > 0))
print("distance positive:", np.all(p_dL > 0))

chirp finite: True
q finite: True
distance finite: True
chirp positive: True
q positive: True
distance positive: True


In [37]:
mass_support = (
    (m1_det >= 1)
    & (m1_det <= 1000)
    & (m2_det >= 1)
    & (m2_det <= 1000)
)

print("All samples satisfy mass constraints:",
      np.all(mass_support))

All samples satisfy mass constraints: True


In [40]:
log_prior_sampling = (
    np.log(p_chirp)
    + np.log(p_q)
    + np.log(p_dL)
)

log_prior_sampling[~mass_support] = -np.inf

In [41]:
log_J = np.log(J)

log_pe_prior_mass_z = (
    log_prior_sampling
    + log_J
)

In [42]:
log_p_chirp = chirp_prior.ln_prob(chirp_stored)
log_p_q = q_prior.ln_prob(q)
log_p_dL = distance_prior.ln_prob(dL)

print("chirp finite:", np.all(np.isfinite(log_p_chirp)))
print("q finite:", np.all(np.isfinite(log_p_q)))
print("distance finite:", np.all(np.isfinite(log_p_dL)))

chirp finite: True
q finite: True
distance finite: True


In [43]:
mass_support = (
    (m1_det >= 1)
    & (m1_det <= 1000)
    & (m2_det >= 1)
    & (m2_det <= 1000)
)

print("All samples satisfy mass constraints:",
      np.all(mass_support))

All samples satisfy mass constraints: True


In [44]:
log_prior_sampling_mass_z = (
    log_p_chirp
    + log_p_q
    + log_p_dL
)

log_prior_sampling_mass_z[~mass_support] = -np.inf

In [45]:
log_pe_prior_mass_z = (
    log_prior_sampling_mass_z
    + np.log(J)
)

In [46]:
distance_transformed = (
    log_p_dL
    + np.log(ddL_dz_analytic(z, cosmology).value)
)

In [47]:
log_pz_shape = (
    np.log(
        cosmology.differential_comoving_volume(z).value
    )
    - np.log1p(z)
)

difference = distance_transformed - log_pz_shape

print("std of difference:", np.std(difference))
print("range of difference:",
      np.min(difference),
      np.max(difference))

std of difference: 0.0001619883208584841
range of difference: -17.00885197298612 -17.007835351133618


In [48]:
# Transform Bilby's d_L density back into a z density
distance_transformed = (
    np.log(p_dL)
    + np.log(ddL_dz_analytic(z, cosmology).value)
)

# UniformSourceFrame expected redshift dependence:
# p(z) ∝ [dVc/dz] / (1+z)
log_pz_shape = (
    np.log(
        cosmology.differential_comoving_volume(z).value
    )
    - np.log1p(z)
)

difference = distance_transformed - log_pz_shape

print("All finite:", np.all(np.isfinite(difference)))
print("std of difference:", np.std(difference))
print("min difference:", np.min(difference))
print("max difference:", np.max(difference))
print("range:", np.ptp(difference))


All finite: True
std of difference: 0.0001619883208584841
min difference: -17.00885197298612
max difference: -17.007835351133618
range: 0.0010166218525000659


In [50]:
J_mass_det = (
    q**(3/5)
    / (m1_det * (1 + q)**(1/5))
)

log_mass_transformed = (
    log_p_chirp
    + log_p_q
    + np.log(J_mass_det)
)

print("All finite:", np.all(np.isfinite(log_mass_transformed)))
print("std:", np.std(log_mass_transformed))
print("min:", np.min(log_mass_transformed))
print("max:", np.max(log_mass_transformed))
print("range:", np.ptp(log_mass_transformed))

All finite: True
std: 3.1753708693840066e-16
min: -0.5083172203826751
max: -0.5083172203826729
range: 2.220446049250313e-15


In [51]:
log_pe_prior_mass_z = (
    log_p_chirp
    + log_p_q
    + log_p_dL
    + np.log(J)
)

log_pe_prior_mass_z[~mass_support] = -np.inf

print("All finite on support:",
      np.all(np.isfinite(log_pe_prior_mass_z[mass_support])))

print("min:", np.min(log_pe_prior_mass_z[mass_support]))
print("median:", np.median(log_pe_prior_mass_z[mass_support]))
print("max:", np.max(log_pe_prior_mass_z[mass_support]))

All finite on support: True
min: -1.2549831366592965
median: 2.2198031074357782
max: 2.854056480127734
